# 177 — IA para educación y aprendizaje adaptativo

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución 1 — BKT a mano

a)
```text
P(L|correcto) = 0.30·0.95 / (0.285 + 0.70·0.25) = 0.285/0.46 ≈ 0.6196
P(L')         = 0.6196 + 0.3804·0.10 ≈ 0.6576
```

b) Incorrecto desde 0.6576:
```text
P(L|error) = 0.6576·0.05 / (0.0329 + 0.3424·0.75) = 0.0329/0.2897 ≈ 0.1135
P(L')      = 0.1135 + 0.8865·0.10 ≈ 0.2022
```

c) El **error** movió mucho más la estimación (0.658 → 0.202, caída de 0.46; el
acierto subió 0.30 → 0.62, +0.32... comparable pero menor en razón de verosimilitud):
con S=0.05, fallar dominando es rarísimo, así que un error es evidencia casi
concluyente; con G=0.25, acertar adivinando es plausible y el acierto informa menos.


In [ ]:
L, S, G, T = 0.30, 0.05, 0.25, 0.10
post_c = L * (1 - S) / (L * (1 - S) + (1 - L) * G)
Lp = post_c + (1 - post_c) * T
post_e = Lp * S / (Lp * S + (1 - Lp) * (1 - G))
Lp2 = post_e + (1 - post_e) * T
print(round(post_c, 4), round(Lp, 4), round(post_e, 4), round(Lp2, 4))
assert abs(Lp - 0.6576) < 1e-3 and abs(Lp2 - 0.2022) < 1e-3


## Solución 2 — Aciertos hasta dominar

b) Con G=0.20 se cruza 0.95 en **3 aciertos consecutivos**: la traza es
0.20 → 0.600 → 0.890 → 0.977 (cada paso aplica Bayes por el acierto y luego la
transición P(T)).

c) Con G=0.45 hacen falta **5** (0.20 → 0.433 → 0.664 → 0.828 → 0.920 → 0.965):
cada acierto informa poco porque adivinar es casi tan probable como saber. Lección
de diseño: ítems muy adivinables (verdadero/falso, opciones débiles) alargan la
estimación y retrasan al estudiante; bajar G con distractores de calidad vale tanto
como mejorar el modelo.


In [ ]:
def aciertos_hasta(L, S, G, T, umbral=0.95):
    n = 0
    while L <= umbral and n < 50:
        post = L * (1 - S) / (L * (1 - S) + (1 - L) * G)
        L = post + (1 - post) * T
        n += 1
    return n

print(aciertos_hasta(0.20, 0.10, 0.20, 0.15))  # 3
print(aciertos_hasta(0.20, 0.10, 0.45, 0.15))  # 5
assert aciertos_hasta(0.20, 0.10, 0.20, 0.15) < aciertos_hasta(0.20, 0.10, 0.45, 0.15)


## Solución 3 — Política ZPD

a) En banda: **B (0.75) y C (0.62)**. Elección razonable: B si se quiere consolidar
con señal de éxito frecuente, C si se quiere máxima exigencia dentro de la zona; una
política habitual apunta al centro (~0.7), favoreciendo B.

b) Casi nada: con éxito esperado 0.30, el fallo era el resultado previsto tanto si
domina como si no — la verosimilitud de ambas hipótesis es parecida, la
actualización bayesiana es pequeña, y además se pagó frustración. Ítems fuera de la
ZPD son caros en moral y pobres en información.

c) Andamiaje para `3x + 5 = 20`:
   - Pista 1: "¿Qué operación deshace el +5? Aplícala a los dos lados."
   - Pista 2: "Te queda `3x = 15`. ¿Qué operación deja la x sola?"
   (Nunca "x = 5": la solución elimina la práctica.)


In [ ]:
items = {"A": 0.95, "B": 0.75, "C": 0.62, "D": 0.30}
zpd = {k: v for k, v in items.items() if 0.6 <= v <= 0.8}
eleccion = min(zpd, key=lambda k: abs(zpd[k] - 0.7))
print(zpd, "->", eleccion)
assert set(zpd) == {"B", "C"} and eleccion == "B"


## Solución 4 — Evaluar al tutor

a) Gain X = 58 − 52 = **+6 puntos**; gain Y = 71 − 51 = **+20 puntos**. **Y** es
mejor tutor con diferencia.

b) X probablemente es complaciente: da la solución al primer pedido, valida todo,
evita el esfuerzo — la experiencia es agradable (4.8) pero elimina la práctica
recuperativa y el error productivo que producen aprendizaje. Y exige trabajo dentro
de la ZPD: menos cómodo (3.9), más efectivo.

c) Falta un **grupo de control** (mismo material sin tutor, o instrucción
tradicional) con asignación aleatoria: sin él, el gain podría deberse al paso del
tiempo, al efecto del propio pre-test o a la autoselección de estudiantes. Es el
diseño con el que se midió el 2σ original.
